In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType, TimestampType, FloatType

In [0]:
catalog_name='ecommerce'


In [0]:
df=spark.table(f"{catalog_name}.silver.slv_order_items")
display(df.limit(10))

In [0]:
# 1) add gross amount
df=df.withColumn(
    "gross_amount",
    F.col("quantity")*F.col("unit_price")
)

#2) Add discount_amount (discount_pct is already numeric, eg. 21 -> 21%)
df=df.withColumn(
    "discount_amount",
    F.ceil(F.col("gross_amount")*(F.col("discount_pct")/100.0))
)

# 3) add sales_amount = gross -discount + tax
df=df.withColumn(
    "sales_amount",
    F.col("gross_amount") - F.col("discount_amount")+F.col("tax_amount")
)

# 4) add date_id
df=df.withColumn("date_id", F.date_format(F.col("dt"),"yyyyMMdd").cast(IntegerType())) # create date_key

# Coupon flag
# coupon flag = 1 if coupon is not null else 0
df=df.withColumn(
    "coupon_flag",
    F.when(F.col("coupon_code").isNotNull(), F.lit(1))
    .otherwise(F.lit(0))
)

#Display the final changes
display(df.limit(5))

In [0]:
# 1) Define your fixed FX rates (as of now 2025-20-15, like your PBI note) ---
# right now we are using hard coded exchage rate bu in case of real time project we can use API to get the latest rates
fx_rate={
    "INR":1.00,
    "AED":24.18,
    "AUD":57.55,
    "CAD":62.93,
    "GBP":117.98,
    "SGD":68.18,
    "USD":88.29

}

rates =[(k,float(v)) for k,v in fx_rate.items()]
rate_df=spark.createDataFrame(rates,["currency","inr_rate"])

display(rate_df)

In [0]:
df= (
    df
    .join(
        rate_df,
        rate_df.currency == F.upper(F.trim(F.col("unit_price_currency"))),
        "left"
    )
    .withColumn("sale_amount_inr",F.col("sales_amount") * F.col("inr_rate"))
    .withColumn("tax_amount_inr",F.col("tax_amount")*F.col("inr_rate"))
    .withColumn("discount_amount_inr",F.col("tax_amount")*F.col("inr_rate"))
)

In [0]:
orders_gold_df=df.select(
    F.col("date_id"),
    F.col("dt").alias("transaction_date"),
    F.col("order_ts").alias("transaction_ts"),
    F.col("customer_id"),
    F.col("item_seq"),
    F.col("product_id"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("coupon_flag"),
    F.col("unit_price_currency"),
    F.col("quantity"),
    F.col("unit_price"),
    F.col("gross_amount"),
    F.col("discount_pct"),
    F.col("discount_amount"),
    F.col("discount_amount_inr"),
    F.col("tax_amount"),
    F.col("tax_amount_inr"),
    F.col("sales_amount").alias("net_amount"),
    F.col("sale_amount_inr").alias("net_amount_inr")

)

In [0]:
orders_gold_df.limit(5).display()

In [0]:
#write raw data to the layer (catalog ecommers schema gold, table : gld_brands)
orders_gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeShema","true") \
    .saveAsTable(f"{catalog_name}.gold.gld_fact_order_items")

In [0]:
spark.sql(f"SELECT count(*) FROM {catalog_name}.gold.gld_fact_order_items").show()